<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Principle Component Analysis - Cancer
</b></font> </br></p>

---


# 0 | Install & Import
---

In [ ]:
# Install

In [ ]:
# Daten
import numpy as np
from pandas import read_csv, DataFrame

# Dimensionality Reduction
from sklearn.decomposition import PCA

# Modell (Klassifikation)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Visualisierung
import plotly.express as px

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1 | Understand
---

In [ ]:
df = read_csv('https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/breast_cancer_wisconsin.csv')

In [ ]:
data = df.copy()
target = data.pop("Class")

# 2 | Prepare
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

In [ ]:
# drop na
data = data.dropna()
target = target[data.index]

In [ ]:
data

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Validiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

**In der Praxis relevant wenn:**
- Hochdimensionale Daten visualisiert werden sollen (Reduktion auf 2–3 Dimensionen)
- Multikollinearität in den Features reduziert werden soll (korrelierte Features zusammenfassen)
- Die Trainingsdaten sehr viele Features haben und Overfitting verhindert werden soll

**Nicht geeignet wenn:**
- Interpretierbarkeit der Features zentral ist → PCA-Komponenten sind schwer erklärbar
- Nichtlineare Strukturen in den Daten vorliegen → dann t-SNE oder UMAP verwenden

**Typischer Fehler:**
PCA vor dem Train-Test-Split auf allen Daten anwenden — das führt zu Data Leakage. PCA immer nur auf Trainingsdaten fitten, dann auf Test-Daten transformieren.

In [ ]:
import base64
from IPython.display import Image, display

diagram = """
flowchart LR
    DATA[Rohdaten] --> PCA["PCA<br/>fit_transform"]
    PCA --> EV["Erklärte Varianz<br/>analysieren"]
    EV --> OPT["Optimale Anzahl<br/>Komponenten wählen"]
    OPT --> RED["Reduzierter<br/>Datensatz"]
    RED --> VIS["Visualisierung /<br/>Downstream-Modell"]
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=750))

In [ ]:
n_components = data.shape[1]
model = PCA(n_components=n_components)

In [ ]:
pca_np = model.fit_transform(data)
pca_df = DataFrame(pca_np)

In [ ]:
pca_df['target'] = target.values

In [ ]:
# pca_df.to_csv('pca_data.csv', index=False)
# print("pca_data.csv has been saved. You can download it from the file browser on the left.")

# 4 | Evaluate
---

<p><font color='black' size="5">
Erklärte Varianz
</font></p>

Die erklärte Varianz ist der Anteil der Gesamtvarianz im Datensatz, der durch eine bestimmte Hauptkomponente oder eine Menge von Hauptkomponenten erklärt wird. Sie ist ein Maß dafür, wie gut die Hauptkomponenten die Daten repräsentieren.

**PCA**

PCA ist eine Technik zur Dimensionsreduktion, die verwendet wird, um die Dimensionalität eines Datensatzes zu reduzieren, indem er in einen niedrigerdimensionalen Raum transformiert wird. Dies geschieht durch die Suche nach den Hauptkomponenten des Datensatzes, die die Richtungen der größten Varianz in den Daten darstellen. Die erste Hauptkomponente erklärt den größten Teil der Varianz, die zweite Hauptkomponente den zweitgrößten Teil der Varianz und so weiter.

**Erklärte Varianz in PCA**

Im Zusammenhang mit PCA wird die erklärte Varianz verwendet, um zu bestimmen, wie viele Hauptkomponenten im niedrigerdimensionalen Raum beibehalten werden sollen. Ziel ist es, so viele Hauptkomponenten beizubehalten, dass ein großer Teil der Gesamtvarianz erklärt wird, ohne zu viele Dimensionen beizubehalten.

Die erklärte Varianz jeder Hauptkomponente kann berechnet werden, indem das Eigenwert der Hauptkomponente durch die Summe der Eigenwerte aller Hauptkomponenten dividiert wird. Die Gesamtvarianz, die durch eine Menge von Hauptkomponenten erklärt wird, ist die Summe der erklärten Varianzen jeder Hauptkomponente.

**Beispiel**

Angenommen, ein Datensatz hat 10 Dimensionen und die PCA wird verwendet, um ihn auf 2 Dimensionen zu reduzieren. Die erste Hauptkomponente erklärt 70 % der Gesamtvarianz und die zweite Hauptkomponente erklärt 20 % der Gesamtvarianz. Zusammen erklären diese beiden Hauptkomponenten 90 % der Gesamtvarianz. Dies bedeutet, dass der Datensatz in einem 2-dimensionalen Raum mit einem Verlust von nur 10 % der Gesamtvarianz repräsentiert werden kann.

**Schlussfolgerung**

Die erklärte Varianz ist ein wichtiges Konzept in PCA. Sie wird verwendet, um zu bestimmen, wie viele Hauptkomponenten im niedrigerdimensionalen Raum beibehalten werden sollen. Ziel ist es, so viele Hauptkomponenten beizubehalten, dass ein großer Teil der Gesamtvarianz erklärt wird, ohne zu viele Dimensionen beizubehalten.

In [ ]:
opt_point = 90.0  # erklärte Varianz sollte >= 90% sein
explained_variance = [round(v, 3) for v in model.explained_variance_ratio_ * 100]
cum_expl_variance = np.cumsum(model.explained_variance_ratio_ * 100)
pc_greater = np.argmax(cum_expl_variance >= opt_point) + 1

In [ ]:
# Ausgabe "erklärte Varianz"
print("-" * 40)
print("Erklärte Varianz (%):\n", DataFrame(explained_variance))

print("-" * 40)
print("Kum. erkl. Varianz (%):\n",DataFrame(cum_expl_variance))

print("-" * 40)
print(f"Optimaler Punkt: {pc_greater}", "\n")

In [ ]:
# @title
# @markdown   <p><font size="4" color='green'>  Plot-Funktion</font> </br></p>
def plot_explained_variance(data, cum_sum, opt_point, d):
    # Erstelle ein DataFrame für die Daten
    df = DataFrame(
        {
            "Principal Component": range(1, data.shape[1] + 1),
            "Cumulative Explained Variance": cum_sum,
        }
    )

    # Erstelle das Liniendiagramm mit plotly-express
    fig = px.line(
        df,
        x="Principal Component",
        y="Cumulative Explained Variance",
        title="Explained Variance by Principal Components",
        labels={"Cumulative Explained Variance": "Cumulative Explained Variance"},
        width=700,
        height=400,
        markers=True,
    )

    # Füge den optimalen Punkt als Scatter hinzu
    fig.add_scatter(
        x=[d],
        y=[cum_sum[d - 1]],
        mode="markers",
        marker=dict(color="red", size=10),
        name="Opt. Point >" + str(opt_point) + "%",
    )

    # Zeige die Grafik an
    fig.show()

In [ ]:
# Grafik "erklärte Varianz"
plot_explained_variance(data, cum_expl_variance, opt_point, pc_greater)

<p><font color='black' size="5">
Gewichtung Features in den Hauptkomponenten
</font></p>

Die Werte in pca.components_ stellen die Gewichte oder Koeffizienten der ursprünglichen Features in den jeweiligen Hauptkomponenten dar.

Jede Zeile der Matrix entspricht einer Hauptkomponente.
Jede Spalte der Matrix entspricht einem ursprünglichen Feature.
Der Wert an der Position (i, j) in der Matrix gibt das Gewicht des Features j in der Hauptkomponente i an.
Interpretation:

+ Vorzeichen: Das Vorzeichen eines Gewichts (+ oder -) gibt die Richtung des Einflusses des Features auf die Hauptkomponente an. Ein positives Gewicht bedeutet, dass ein Anstieg des Feature-Werts zu einem Anstieg des Werts der Hauptkomponente führt. Ein negatives Gewicht bedeutet, dass ein Anstieg des Feature-Werts zu einem Abfall des Werts der Hauptkomponente führt.

+ Betrag: Der Betrag eines Gewichts gibt die Stärke des Einflusses des Features auf die Hauptkomponente an. Je größer der absolute Wert des Gewichts, desto stärker trägt das entsprechende Feature zur Hauptkomponente bei.

Zusammenfassend:

+ Hohe positive Werte: Das Feature trägt stark und positiv zur Hauptkomponente bei.
+ Hohe negative Werte: Das Feature trägt stark und negativ zur Hauptkomponente bei.
+ Werte nahe Null: Das Feature hat wenig Einfluss auf die Hauptkomponente.

**Beispiel:**

Nehmen wir an, wir haben einen Datensatz mit zwei Features: "Größe" und "Gewicht". Wenn die PCA eine Hauptkomponente findet, die "PCA1" genannt wird, und das Feature "Größe" einen starken und negativen Beitrag zu dieser Hauptkomponente hat, bedeutet dies, dass größere Personen tendenziell einen niedrigeren Wert auf der "PCA1"-Hauptkomponente haben.

In [ ]:
feature = data.columns
DataFrame(model.components_, columns=feature)

<p><font color='black' size="5">
Gewichtung Features in Hauptkomponente 1
</font></p>

In [ ]:
# Feature Importance
title_ = "Gewichtung Feature in Hauptkomponente 1"
px.bar(
    x=model.components_[0], y=data.columns, title=title_, width=800, height=600
).update_yaxes(categoryorder="total ascending")

In [ ]:
# Feature Importance
title_ = "Gewichtung Feature in Hauptkomponente 2"
px.bar(
    x=model.components_[1], y=data.columns, title=title_, width=800, height=600
).update_yaxes(categoryorder="total ascending")

<p><font color='black' size="5">
Interpretation Gewichtung
</font></p>

Am Beispiel eines Brustkrebs-Datensatzes mit neun Zellmerkmalen kann man zeigen, wie PCA mehrere **ähnliche** Messgrößen zu wenigen aussagekräftigen Komponenten bündelt.

**Grundidee**

Die neun ursprünglichen Merkmale (z. B. Zellgröße, Zellform, nackte Kerne) erfassen teilweise dieselben Eigenschaften [1]. Anstatt alle Merkmale separat zu verwenden, projiziert die PCA die Daten auf neue Achsen. **Bereits die ersten beiden Hauptkomponenten (PC1 und PC2) enthalten einen Großteil der relevanten Information.**



**Beispiel: Cancer**

Die Hauptkomponenten lassen sich anhand ihrer Ladungen (Gewichte) inhaltlich interpretieren:

+ PC1 – Zellgleichmäßigkeit      
Diese Komponente wird stark durch Uniformity of Cell Size (0,40), Uniformity of Cell Shape (0,39) und Bare Nuclei (0,44) geprägt. Alle Beiträge sind positiv, sodass hohe PC1-Werte eine starke Ausprägung dieser Merkmale anzeigen.
In den Daten sind hohe PC1-Werte überwiegend mit bösartigen Zellen (target = 4) assoziiert, während niedrige PC1-Werte häufiger bei gutartigen Zellen (target = 2) auftreten.

+ PC2 – Kerndominanz      
PC2 kontrastiert stark ausgeprägte nackte Kerne (+0,78) mit ungleichmäßiger Zellform und -größe (–0,23 / –0,16). Die Trennwirkung zwischen gutartigen und bösartigen Zellen ist hier deutlich schwächer als bei PC1 und ergänzt die erste Hauptkomponente.

**Nutzen**

Durch die Reduktion von neun Merkmalen auf zwei unabhängige Hauptkomponenten bleibt die wesentliche Struktur der Daten erhalten. Gleichzeitig werden Visualisierung, Klassifikation und weitere Analysen deutlich vereinfacht.

**Hinweis:**
Die inhaltliche Benennung der Hauptkomponenten ist eine Interpretation, keine Eigenschaft der PCA selbst. Sie hängt vom Datensatz und den jeweiligen Ladungen ab und sollte immer *vorsichtig* vorgenommen werden.

Fußnote:       
[1] *Man misst mehrere Merkmale, obwohl sie **Ähnliches** erfassen, um Robustheit, Objektivität und Trennschärfe zu gewinnen.*

# 5 | Deploy
---

# A | Feature Importance Decision Tree
---

In [ ]:
data_train_dt, data_test_dt, target_train_dt, target_test_dt = train_test_split(
    data, target, test_size=0.2, random_state=42, stratify=target
)

In [ ]:
# Decision Tree
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(data_train_dt, target_train_dt)

In [ ]:
title_ = "Feature Importance Cancer"
px.bar(
    x=dt_model.feature_importances_, y=data.columns, title=title_, width=1000, height=600
).update_yaxes(categoryorder="total ascending")